***Install Dependencies***

In [0]:
%pip install openai==1.30.0 anthropic==0.28.0 mlflow==2.13.0 scikit-learn pandas httpx==0.27.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


***Setup Keys and Load Data***

In [0]:
import os
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# API Keys
os.environ["OPENAI_API_KEY"] = "DUMMY KEY" 
os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get(scope="aai510", key="anthropic_api_key")

# Load cities data
cities_pd = spark.read.table("main.alcala_voyages.clean_europe_cities").toPandas()

# Load hotel aggregated data
hotels_pd = spark.read.table("main.alcala_voyages.clean_aggregated_hotels").toPandas()

# Rebuild TF-IDF index directly — no tmp files needed
print("Building TF-IDF index...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(
    hotels_pd['Hotel_Full_Description'].fillna('')
)

print(f"Cities loaded: {len(cities_pd)}")
print(f"Hotels loaded: {len(hotels_pd)}")
print(f"TF-IDF index built on {len(hotels_pd)} hotels!")

Building TF-IDF index...
Cities loaded: 69
Hotels loaded: 1494
TF-IDF index built on 1494 hotels!


***Define the 4 Tools***

In [0]:
import json

def search_destinations(travel_style: str = None, budget: str = None, country: str = None) -> str:
    """Search European cities by travel style, budget, or country."""
    results = cities_pd.copy()
    
    if country:
        results = results[results['country'].str.lower().str.contains(country.lower(), na=False)]
    
    if budget:
        results = results[results['budget_level'].str.lower().str.contains(budget.lower(), na=False)]
    
    if travel_style and travel_style.lower() in results.columns:
        results = results.sort_values(travel_style.lower(), ascending=False)
    
    results = results.head(5)
    
    if results.empty:
        return "No destinations found matching your criteria."
    
    output = []
    for _, row in results.iterrows():
        output.append(f"- {row['city']}, {row['country']} | Budget: {row.get('budget_level', 'N/A')}")
    
    return "Top destination recommendations:\n" + "\n".join(output)


def get_climate_info(city: str, month: str = None) -> str:
    """Get climate information for a European city."""
    result = cities_pd[cities_pd['city'].str.lower().str.contains(city.lower(), na=False)]
    
    if result.empty:
        return f"No climate data found for {city}."
    
    row = result.iloc[0]
    city_name = row['city']
    country = row['country']
    
    # Look for temperature columns
    temp_cols = [col for col in cities_pd.columns if 'temp' in col.lower() or 'climate' in col.lower()]
    
    if temp_cols:
        temps = {col: row[col] for col in temp_cols if pd.notna(row[col])}
        temp_info = ", ".join([f"{k}: {v}" for k, v in list(temps.items())[:4]])
        return f"Climate info for {city_name}, {country}: {temp_info}"
    else:
        return f"{city_name}, {country} is located in Europe. Best travel months are generally April-October."


def recommend_hotels(query: str, top_n: int = 3) -> str:
    """Find hotels using TF-IDF semantic search on guest reviews."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[-top_n:][::-1]
    
    results = hotels_pd.iloc[top_indices]
    
    output = []
    for _, row in results.iterrows():
        score = round(row.get('Final_Average_Score', 0), 1)
        output.append(f"- {row['Hotel_Name']} | {row['Hotel_Address']} | Rating: {score}/10")
    
    return "Recommended hotels based on your preferences:\n" + "\n".join(output)


def reject_out_of_scope(query: str) -> str:
    """Reject queries outside the travel planning scope."""
    return (
        f"I'm sorry, I can't help with '{query}'. "
        "I am Alcalá Voyages' European travel planning assistant. "
        "I can help you with destination recommendations, climate information, "
        "and hotel suggestions for European travel. "
        "Please ask me about planning your European trip!"
    )


# Tool registry
TOOLS = {
    "search_destinations": search_destinations,
    "get_climate_info": get_climate_info,
    "recommend_hotels": recommend_hotels,
    "reject_out_of_scope": reject_out_of_scope
}

print("All 4 tools defined successfully!")

All 4 tools defined successfully!


***Define the ReAct Agent***

In [0]:
import openai
import anthropic
import mlflow
import json
import re

# Out of scope keywords
OUT_OF_SCOPE_KEYWORDS = [
    "flight", "visa", "medical", "insurance", "stock", "weather forecast",
    "restaurant reservation", "car rental", "cruise", "train ticket"
]

def is_out_of_scope(query: str) -> bool:
    return any(kw in query.lower() for kw in OUT_OF_SCOPE_KEYWORDS)

# Tool definitions for LLM function calling
tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "search_destinations",
            "description": "Search European cities by travel style, budget level, or country",
            "parameters": {
                "type": "object",
                "properties": {
                    "travel_style": {"type": "string", "description": "Travel style e.g. beaches, culture, adventure"},
                    "budget": {"type": "string", "description": "Budget level: Luxury, Mid-range, or Budget"},
                    "country": {"type": "string", "description": "European country name"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_climate_info",
            "description": "Get climate and temperature information for a European city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "month": {"type": "string", "description": "Month of travel"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_hotels",
            "description": "Recommend hotels based on traveler preferences using guest review search",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Hotel preferences e.g. romantic, family friendly, central location"},
                    "top_n": {"type": "integer", "description": "Number of hotels to return"}
                },
                "required": ["query"]
            }
        }
    }
]

def run_agent_gpt4o(user_query: str) -> dict:
    """Run the ReAct agent using GPT-4o."""
    
    # Check out of scope first
    if is_out_of_scope(user_query):
        return {
            "llm": "GPT-4o",
            "query": user_query,
            "response": reject_out_of_scope(user_query),
            "tools_called": ["reject_out_of_scope"],
            "out_of_scope": True
        }
    
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    messages = [
        {"role": "system", "content": """You are the Alcalá Voyages AI travel planning assistant, 
        specializing in European travel. Help users plan their European trips by recommending 
        destinations, providing climate information, and suggesting hotels. 
        Always use the available tools to provide grounded recommendations.
        Be friendly, specific, and helpful."""},
        {"role": "user", "content": user_query}
    ]
    
    tools_called = []
    max_iterations = 5
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tool_definitions,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        # If no tool calls, we have our final answer
        if not message.tool_calls:
            return {
                "llm": "GPT-4o",
                "query": user_query,
                "response": message.content,
                "tools_called": tools_called,
                "out_of_scope": False
            }
        
        # Process tool calls
        messages.append(message)
        
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tools_called.append(tool_name)
            
            # Execute the tool
            if tool_name in TOOLS:
                tool_result = TOOLS[tool_name](**tool_args)
            else:
                tool_result = f"Tool {tool_name} not found"
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })
    
    return {
        "llm": "GPT-4o",
        "query": user_query,
        "response": "Max iterations reached",
        "tools_called": tools_called,
        "out_of_scope": False
    }


def run_agent_claude(user_query: str) -> dict:
    """Run the ReAct agent using Claude Sonnet 4."""
    
    # Check out of scope first
    if is_out_of_scope(user_query):
        return {
            "llm": "Claude Sonnet 4",
            "query": user_query,
            "response": reject_out_of_scope(user_query),
            "tools_called": ["reject_out_of_scope"],
            "out_of_scope": True
        }
    
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    
    # Claude tool format
    claude_tools = [
        {
            "name": "search_destinations",
            "description": "Search European cities by travel style, budget level, or country",
            "input_schema": {
                "type": "object",
                "properties": {
                    "travel_style": {"type": "string", "description": "Travel style e.g. beaches, culture, adventure"},
                    "budget": {"type": "string", "description": "Budget level: Luxury, Mid-range, or Budget"},
                    "country": {"type": "string", "description": "European country name"}
                }
            }
        },
        {
            "name": "get_climate_info",
            "description": "Get climate and temperature information for a European city",
            "input_schema": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "month": {"type": "string", "description": "Month of travel"}
                },
                "required": ["city"]
            }
        },
        {
            "name": "recommend_hotels",
            "description": "Recommend hotels based on traveler preferences using guest review search",
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Hotel preferences"},
                    "top_n": {"type": "integer", "description": "Number of hotels to return"}
                },
                "required": ["query"]
            }
        }
    ]
    
    messages = [{"role": "user", "content": user_query}]
    tools_called = []
    max_iterations = 5
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=2048,
            system="""You are the Alcalá Voyages AI travel planning assistant, 
            specializing in European travel. Help users plan their European trips by recommending 
            destinations, providing climate information, and suggesting hotels. 
            Always use the available tools to provide grounded recommendations.
            Be friendly, specific, and helpful.""",
            messages=messages,
            tools=claude_tools
        )
        
        # Check if done
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, 'text'):
                    final_text += block.text
            return {
                "llm": "Claude Sonnet 4",
                "query": user_query,
                "response": final_text,
                "tools_called": tools_called,
                "out_of_scope": False
            }
        
        # Process tool use
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name
                tool_args = block.input
                tools_called.append(tool_name)
                
                if tool_name in TOOLS:
                    tool_result = TOOLS[tool_name](**tool_args)
                else:
                    tool_result = f"Tool {tool_name} not found"
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": tool_result
                })
        
        # Add assistant response and tool results to messages
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    return {
        "llm": "Claude Sonnet 4",
        "query": user_query,
        "response": "Max iterations reached",
        "tools_called": tools_called,
        "out_of_scope": False
    }

print("Agent functions defined successfully!")

Agent functions defined successfully!


***The LLM judge***

In [0]:
def llm_judge(query: str, response: str, llm_name: str) -> dict:
    """Score agent response using GPT-4o as judge."""
    
    client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    judge_prompt = f"""You are an expert evaluator for an AI travel planning assistant.
    
User Query: {query}
Agent Response ({llm_name}): {response}

Score this response on three dimensions from 1-5:
1. Relevance: Does the response directly address the user's travel query?
2. Accuracy: Are the recommendations grounded and specific (not vague)?
3. Helpfulness: Would a real traveler find this response useful and actionable?

Respond in this exact JSON format:
{{
    "relevance": <1-5>,
    "accuracy": <1-5>,
    "helpfulness": <1-5>,
    "rationale": "<brief explanation>"
}}"""

    judge_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0
    )
    
    try:
        import json
        content = judge_response.choices[0].message.content
        # Strip markdown if present
        content = content.replace("```json", "").replace("```", "").strip()
        scores = json.loads(content)
    except:
        scores = {"relevance": 0, "accuracy": 0, "helpfulness": 0, "rationale": "Parse error"}
    
    return scores

print("LLM judge defined!")

LLM judge defined!


***Run All 5 Traces with MLflow***

In [0]:
import mlflow
import json

# Get username for MLflow path
username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/alcala_voyages_agent")
print(f"MLflow experiment set for: {username}")

# Define 5 test queries
test_queries = [
    "I want to visit a cultural city in Spain with a mid-range budget. What do you recommend?",
    "What is the climate like in Paris in July?",
    "I need a romantic hotel in London with great reviews.",
    "Can you help me book a flight to Rome?",  # Out of scope
    "I want adventure activities in Austria. What cities and hotels do you suggest?"  # Multi-LLM comparison
]

all_results = []

for i, query in enumerate(test_queries):
    print(f"\n{'='*60}")
    print(f"TRACE {i+1}: {query}")
    print('='*60)
    
    with mlflow.start_run(run_name=f"trace_{i+1}"):
        
        mlflow.log_param("query", query)
        mlflow.log_param("trace_number", i+1)
        
        # Run GPT-4o
        gpt_result = run_agent_gpt4o(query)
        print(f"\nGPT-4o Response:\n{gpt_result['response']}")
        print(f"Tools called: {gpt_result['tools_called']}")
        
        # Judge GPT-4o
        gpt_scores = llm_judge(query, gpt_result['response'], "GPT-4o")
        
        mlflow.log_param("llm_gpt4o", "gpt-4o")
        mlflow.log_metric("gpt4o_relevance", gpt_scores.get('relevance', 0))
        mlflow.log_metric("gpt4o_accuracy", gpt_scores.get('accuracy', 0))
        mlflow.log_metric("gpt4o_helpfulness", gpt_scores.get('helpfulness', 0))
        mlflow.log_metric("gpt4o_avg_score", 
                         (gpt_scores.get('relevance', 0) + 
                          gpt_scores.get('accuracy', 0) + 
                          gpt_scores.get('helpfulness', 0)) / 3)
        
        print(f"\nGPT-4o Judge Scores: {gpt_scores}")
        
        # Trace 5 only — run Claude too for multi-LLM comparison
        if i == 4:
            print(f"\nRunning Claude Sonnet 4 on same query for comparison...")
            claude_result = run_agent_claude(query)
            print(f"\nClaude Sonnet 4 Response:\n{claude_result['response']}")
            print(f"Tools called: {claude_result['tools_called']}")
            
            claude_scores = llm_judge(query, claude_result['response'], "Claude Sonnet 4")
            
            mlflow.log_param("llm_claude", "claude-sonnet-4-5")
            mlflow.log_metric("claude_relevance", claude_scores.get('relevance', 0))
            mlflow.log_metric("claude_accuracy", claude_scores.get('accuracy', 0))
            mlflow.log_metric("claude_helpfulness", claude_scores.get('helpfulness', 0))
            mlflow.log_metric("claude_avg_score",
                            (claude_scores.get('relevance', 0) + 
                             claude_scores.get('accuracy', 0) + 
                             claude_scores.get('helpfulness', 0)) / 3)
            
            print(f"\nClaude Judge Scores: {claude_scores}")
            print(f"\n--- MULTI-LLM COMPARISON ---")
            print(f"GPT-4o avg: {round((gpt_scores.get('relevance',0) + gpt_scores.get('accuracy',0) + gpt_scores.get('helpfulness',0))/3, 2)}")
            print(f"Claude avg: {round((claude_scores.get('relevance',0) + claude_scores.get('accuracy',0) + claude_scores.get('helpfulness',0))/3, 2)}")
        
        all_results.append({
            "trace": i+1,
            "query": query,
            "gpt4o_response": gpt_result['response'],
            "gpt4o_tools": gpt_result['tools_called'],
            "gpt4o_scores": gpt_scores
        })

print("\n\nAll 5 traces completed and logged to MLflow!")

MLflow experiment set for: ai.dataflow.team@gmail.com

TRACE 1: I want to visit a cultural city in Spain with a mid-range budget. What do you recommend?

GPT-4o Response:
For a cultural trip to Spain on a mid-range budget, I recommend considering the following cities:

1. **Barcelona**: Known for its stunning architecture including Gaudí's masterpieces, vibrant arts scene, and rich history.
2. **Madrid**: The capital city offers world-class museums like the Prado, lively neighborhoods, and a dynamic nightlife.
3. **Seville**: Famous for flamenco dancing, historical sites including the Alcázar, and delicious tapas.
4. **Bilbao**: Home to the impressive Guggenheim Museum and a mix of modern and traditional Basque culture.
5. **Granada**: Renowned for the breathtaking Alhambra Palace, beautiful Moorish architecture, and vibrant street life.

Each city has its unique cultural flair and plenty to explore. If you need more details or hotel recommendations in any of these cities, just let me 

***Performance Commentary***

In [0]:
print("""
=============================================================
AGENT PERFORMANCE COMMENTARY — ALCALÁ VOYAGES TRAVEL AGENT
=============================================================

OVERVIEW:
The Alcalá Voyages ReAct agent was evaluated across 5 traces
using GPT-4o as the primary LLM, with Claude Sonnet 4 compared
on Trace 5. The agent uses 4 tools grounded in real datasets.

TRACE RESULTS:
- Trace 1 (Spain destinations): GPT-4o scored 5/5 across all
  dimensions. search_destinations() returned relevant cities.
- Trace 2 (Paris climate): GPT-4o scored 5/5. get_climate_info()
  returned accurate July temperature data from the dataset.
- Trace 3 (London hotel): GPT-4o scored 4.3/5. recommend_hotels()
  returned The Arch London at 9.1/10 via TF-IDF RAG search.
- Trace 4 (Flight booking): Correctly rejected as out-of-scope.
  Note: LLM judge scored this low (2.3) because it evaluated
  helpfulness from a user perspective, not agent correctness.
  Human review confirms the rejection was appropriate.
- Trace 5 (Austria adventure): Both LLMs scored 3.33/5. Both
  recommended Vienna hotels despite Innsbruck/Hallstatt being
  better adventure bases. This is a known limitation of the
  hotel dataset which skews toward Vienna properties.

MULTI-LLM COMPARISON (Trace 5):
GPT-4o and Claude Sonnet 4 scored identically (3.33/5).
Claude provided more structured markdown formatting and offered
a follow-up question. GPT-4o was more concise. Both had the
same tool calling pattern: search_destinations + recommend_hotels.

ROI ANALYSIS:
- GPT-4o: ~$0.005 per query
- Claude Sonnet 4: ~$0.003 per query (40% cheaper)
- At 500 clients/month: Claude saves ~$1/month at this scale
- Performance delta: 0 (identical scores on Trace 5)
- Recommendation: Claude Sonnet 4 offers equivalent performance
  at lower cost. For simple destination and hotel queries,
  Claude Sonnet 4 is the better ROI choice for Alcalá Voyages.

HUMAN IN THE LOOP:
All 5 traces were manually reviewed by the team. Key finding:
Trace 4 judge scores (2.3) do not reflect agent correctness —
the rejection was appropriate. Human review is essential to
catch cases where the LLM judge misinterprets out-of-scope
handling as unhelpfulness.

AGENT STRENGTHS:
- Climate data was highly accurate (real dataset values)
- Destination search worked well for country/style filtering
- Hotel RAG returned specific named properties with ratings
- Out-of-scope detection worked correctly

AGENT LIMITATIONS:
- Hotel index skews toward Vienna, limiting diversity
- TF-IDF is less powerful than true vector embeddings
- Dataset limited to 6 European countries
- Adventure-specific hotel filtering could be improved

WHAT WE LEARNED:
Building an agent requires careful attention to data quality.
The hotel dataset's geographic skew toward Vienna directly
impacted recommendation quality for non-Vienna queries.
LLM-as-judge evaluation is powerful but needs human validation
especially for edge cases like out-of-scope handling.
=============================================================
""")


AGENT PERFORMANCE COMMENTARY — ALCALÁ VOYAGES TRAVEL AGENT

OVERVIEW:
The Alcalá Voyages ReAct agent was evaluated across 5 traces
using GPT-4o as the primary LLM, with Claude Sonnet 4 compared
on Trace 5. The agent uses 4 tools grounded in real datasets.

TRACE RESULTS:
- Trace 1 (Spain destinations): GPT-4o scored 5/5 across all
  dimensions. search_destinations() returned relevant cities.
- Trace 2 (Paris climate): GPT-4o scored 5/5. get_climate_info()
  returned accurate July temperature data from the dataset.
- Trace 3 (London hotel): GPT-4o scored 4.3/5. recommend_hotels()
  returned The Arch London at 9.1/10 via TF-IDF RAG search.
- Trace 4 (Flight booking): Correctly rejected as out-of-scope.
  Note: LLM judge scored this low (2.3) because it evaluated
  helpfulness from a user perspective, not agent correctness.
  Human review confirms the rejection was appropriate.
- Trace 5 (Austria adventure): Both LLMs scored 3.33/5. Both
  recommended Vienna hotels despite Innsbruck/Hall